In [1]:
#Import Packages
import pandas as pd
import seaborn as sns
import numpy as np
from scipy.stats import norm

import time
import sys
import matplotlib.pyplot as plt
import math
import os

from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import MiniBatchKMeans
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture

Read in the data that we will need.
- raw intestine data df
- all probabilities data

In [ ]:
df = pd.read_csv(r'c:\Users\k1van\Box\PanOrgan\Data\Data_Downloaded\labeled_quantified\2023_Hickey_Nature_01\05_25_HuBMAP_tunit.csv')

In [12]:
cells = df.copy()

In [4]:
probabilities_df = pd.read_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\Mixture Model\all_cells_all_neighborhood_probs')

In [ ]:
allinfo_df = pd.read_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\SO_Project2\intestine_all_information.csv')

In [ ]:
allinfo_df

In [33]:
allinfo_df = allinfo_df.rename(columns={'Stroma_prob': 'Stroma'})

In [35]:
# Save the merged dataframe as a CSV file
allinfo_df.to_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\SO_Project2\intestine_all_information_2.csv')

In [ ]:
allinfo_df = pd.read_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\Mixture Model\intestine_all_information_2.csv')

Subset to a region with a follicle (B006_Descending - Sigmoid)

In [ ]:
filtered_cells = allinfo_df[allinfo_df['unique_region']== 'B006_Descending - Sigmoid'] #can change to any unique_region that you want from the dataset

filtered_cells

In [30]:
filtered_cells = filtered_cells.rename(columns={'Stroma_prob': 'Stroma'})

In [ ]:
filtered_cells

In [ ]:
assigned_info = allinfo_df[['Neighborhood','x','y','unique_region']].copy()
assigned_info['Assigned_Probability'] = allinfo_df.apply(
    lambda row: row[row['Neighborhood']], axis=1
)

assigned_info

In [18]:
df = assigned_info.copy()

In [ ]:
df

Define "bins" of probability - low medium and high for each cell

In [20]:
df['Probability_Level'] = pd.cut(
    df['Assigned_Probability'],
    bins=[-float('inf'), 0.33, 0.66, float('inf')],
    labels=['Low', 'Medium', 'High']
)

In [ ]:
df

In [ ]:
df.reset_index(inplace=True, drop=True)
df

Follow code in SpatialOmics_HierarchicalNeighborhood

In [23]:
#Import Neighborhood Functions and Code
cellhier_path = r"C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\cellhier"
sys.path.append(cellhier_path)
from general import *
from plot_john_2 import *
from knn_graph_neighborhood import *  # Imports everything

In [ ]:
# Define column names that will be used for neighborhood analysis
ks = [50,75,100]  # k=5 means it collects 5 nearest neighbors for each center cell
X = 'x'                  # Variable for the X coordinate
Y = 'y'                  # Variable for the Y coordinate
reg = 'unique_region'         # Variable for the filename or region identifier associated with coordinates
neigh = 'Neighborhood'      # Variable for the neighborhood assignemnt of the cell
cluster_col = 'Probability_Level'  # Variable for cell type/subtype classification
sum_cols = list(df[cluster_col].unique())
# List of columns to keep for analysis
keep_cols = [X, Y, reg, cluster_col]

#Run neighborhood analysis function with radial distance threshold
Neigh = Neighborhoods(df,ks = ks,cluster_col = cluster_col,
                      sum_cols=sum_cols,reg=reg,
                      keep_cols=keep_cols,neigh=neigh, X = X,Y=Y, add_dummies=True)
cluster_name_windows = Neigh.k_windows(distance_max=1000) #Distance threshold for cell neighborhoods in terms of pixels conservative of 100 um

In [ ]:
# Concatenate the original 'cells' DataFrame with dummy variables created from 'cluster_col'
# pd.get_dummies() converts categorical variable(s) into dummy/indicator variables
df = pd.concat([df, pd.get_dummies(df[cluster_col], dtype=int)], axis=1)

# Get unique values from the 'cluster_col' column to use for summarization
sum_cols2 = df[cluster_col].unique()

# Retrieve the values for these unique categories as a NumPy array
# This array can be used for further analysis or operations later for calculating the neighborhoods
values = df[sum_cols2].values

#Choose k value to analyze and pull out from dictionary of stored results of vectors
k = 100
windows2 = cluster_name_windows[k]
#Add cell type column to output windows dataframe
windows2[cluster_col] = df[cluster_col]

#Fill in based on above for the number of clusters you want
n_neighborhoods = 10

#return a name of the column for storing the clusters
neighborhood_name = "neighborhood"+str(k)

# Initialize a dictionary to store the centroids for each value of 'k'
k_centroids = {}

# Initialize a MiniBatchKMeans clustering model
# 'n_clusters' is set to 'n_neighborhoods', which is the desired number of clusters
# 'random_state=0' ensures reproducibility of the results
km = MiniBatchKMeans(n_clusters=n_neighborhoods, random_state=0)

# Perform clustering on the data in 'windows2' using the columns specified in 'sum_cols'
# '.values' converts the DataFrame to a NumPy array, which is the input format for KMeans
labels = km.fit_predict(windows2[sum_cols2].values)

# Store the centroids of the clusters in the 'k_centroids' dictionary, keyed by 'k'
k_centroids[k] = km.cluster_centers_

# Add the cluster labels to the original 'cells' DataFrame
# 'neighborhood_name' is presumably a column name where these labels will be stored
df[neighborhood_name] = labels

# Select the centroids for a specific value of 'k' for plotting
k_to_plot = k
niche_clusters = (k_centroids[k_to_plot])

# Calculate the average cell types across the 'values' array
tissue_avgs = values.mean(axis=0)

# Compute fold change (fc) of cell types abundance within a neighborhood versus that in the tissue
# This involves a log2 transformation of the ratio of (niche_clusters + tissue_avgs) to tissue_avgs
# The ratio is normalized by the sum across each row (axis=1), ensuring that the sum of ratios for each row is 1
fc = np.log2(((niche_clusters + tissue_avgs) / (niche_clusters + tissue_avgs).sum(axis=1, keepdims=True)) / tissue_avgs)

# Convert the fold change array into a pandas DataFrame for each cell type
fc = pd.DataFrame(fc, columns=sum_cols2)

# Create a clustered heatmap using seaborn's clustermap function
# 'fc' DataFrame is used as input
# vmin and vmax set the color scale limits for the heatmap (-3 to 3 in this case)
# cmap='bwr' sets the color palette to blue-white-red
# figsize=(10,10) sets the size of the heatmap
#s = sns.clustermap(fc, vmin=-3, vmax=3, cmap='bwr', figsize=(10,10))
s = sns.clustermap(
    fc,
    vmin=-3,
    vmax=3,
    cmap='bwr',
    figsize=(10, 10),
    xticklabels=True,
    yticklabels=True
)

# Adjust font sizes after plotting
plt.setp(s.ax_heatmap.get_xticklabels(), fontsize=35, rotation=90)
plt.setp(s.ax_heatmap.get_yticklabels(), fontsize=35)

# Optional: set font size for dendrogram labels
s.ax_row_dendrogram.set_ylabel('')
s.ax_col_dendrogram.set_xlabel('')

# Optional: if you want to change colorbar label font size
if s.cax:
    s.cax.tick_params(labelsize=30)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import pairwise_distances
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, leaves_list

# Assuming df, Neighborhoods class, cluster_name_windows etc already exist from your earlier code

# Map Probability_Level to numeric probabilities
probability_map = {
    'High': 0.9,
    'Medium': 0.5,
    'Low': 0.1
}

# Convert to float after mapping
df['assigned_probability'] = df['Probability_Level'].map(probability_map).astype(float)

# Now this will work
high_prob_props = df.groupby(neighborhood_name)['assigned_probability'].apply(
    lambda x: np.mean(x > high_prob_threshold)
)
# Pick k value for clustering neighborhoods
k = 100
windows2 = cluster_name_windows[k]
windows2[cluster_col] = df[cluster_col]

n_neighborhoods = 10
neighborhood_name = "neighborhood" + str(k)

# Perform clustering
km = MiniBatchKMeans(n_clusters=n_neighborhoods, random_state=0)
labels = km.fit_predict(windows2[sum_cols].values)
df[neighborhood_name] = labels

# Store centroids
k_centroids = {k: km.cluster_centers_}
niche_clusters = k_centroids[k]

# Tissue-wide average of each cell type
values = df[sum_cols].values
tissue_avgs = values.mean(axis=0)

# Compute fold change of each cluster vs tissue
fc = np.log2(
    ((niche_clusters + tissue_avgs) / (niche_clusters + tissue_avgs).sum(axis=1, keepdims=True)) / tissue_avgs
)
fc = pd.DataFrame(fc, columns=sum_cols)

# Compute proportion of high-probability cells in each neighborhood cluster
high_prob_threshold = 0.8
high_prob_props = df.groupby(neighborhood_name)['assigned_probability'].apply(lambda x: np.mean(x > high_prob_threshold))

# Reorder dendrogram by increasing proportion of high-probability cells
# Linkage on fold-change data
linkage_matrix = linkage(fc.values, method='average')

# Extract cluster order from dendrogram
original_order = leaves_list(linkage_matrix)

# Sort clusters by high probability proportion
sorted_order = high_prob_props.iloc[original_order].sort_values(ascending=False).index.values

# Reorder fold-change dataframe and high-prob proportion series to match
fc_reordered = fc.loc[sorted_order]
high_prob_props_reordered = high_prob_props.loc[sorted_order]

# Plot heatmap without clustering, using your custom order
sns.set(font_scale=2)
g = sns.heatmap(
    fc_reordered,
    vmin=-3, vmax=3, cmap='bwr',
    yticklabels=True, xticklabels=True
)

# Adjust font sizes for tick labels
plt.setp(s.ax_heatmap.get_xticklabels(), fontsize=35, rotation=90)
plt.setp(s.ax_heatmap.get_yticklabels(), fontsize=35)

# Remove labels for dendrogram axes if desired
s.ax_row_dendrogram.set_ylabel('')
s.ax_col_dendrogram.set_xlabel('')

# Adjust colorbar tick font size
if s.cax:
    s.cax.tick_params(labelsize=30)

# Show the plot
plt.show()

In [ ]:
# Seaborn clustermap for fold change (fc) DataFrame
s = sns.clustermap(
    fc,
    vmin=-3,
    vmax=3,
    cmap='bwr',
    figsize=(10, 10),
    xticklabels=True,
    yticklabels=True
)

# Set x-axis tick label font size and rotate for readability
plt.setp(s.ax_heatmap.get_xticklabels(), fontsize=35, rotation=90)

# Set y-axis tick label font size
plt.setp(s.ax_heatmap.get_yticklabels(), fontsize=35)

# Remove labels for dendrogram axes to clean up the look
s.ax_row_dendrogram.set_ylabel('')
s.ax_col_dendrogram.set_xlabel('')

# Adjust colorbar tick label size
if s.cax:
    s.cax.tick_params(labelsize=30)

# No title set here — clean consistent formatting
plt.show()

In [ ]:
#Subset of clusters to investigate/plot
neigh_list = [0,1,2,3,4,5,6,7,8,9]

#Same code as above, but just with plotting a subset of the clusters
s=sns.clustermap(fc.iloc[neigh_list,:], vmin =-3,vmax = 3,cmap = 'bwr',figsize=(10,4))

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(df.loc[df[neighborhood_name].isin(neigh_list)], X = 'x', Y='y', exp = 'unique_region',
               hue = neighborhood_name, invert_y=False, size = 2, figsize=8)

In [42]:
import seaborn as sns
import matplotlib.colors as mcolors
import json

# Define the neighborhoods and their specific category numbers
neigh_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# Create a custom color palette with a specific number of colors
palette = sns.color_palette("Set1", n_colors=len(neigh_list))

# Map the palette to your category numbers (neighborhood indices)
# If you want to maintain the specific color for each neighborhood, we do it like this:
neigh_palette = {neigh_list[i]: mcolors.rgb2hex(palette[i]) for i in range(len(neigh_list))}

# Save the palette dictionary to a JSON file
with open('neighborhood_palette.json', 'w') as f:
    json.dump(neigh_palette, f)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Define the neighborhoods
neigh_list = [0,1,2,3,4,5,6,7,8,9]

# Create a custom color palette for the neighborhoods
# For example, you can use a palette with a specified number of colors
palette = sns.color_palette("Set1", n_colors=len(neigh_list))

# Subset of clusters to investigate/plot
s = sns.clustermap(fc.iloc[neigh_list,:], vmin=-3, vmax=3, cmap='bwr', figsize=(10, 4))

# Modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(
    df.loc[df[neighborhood_name].isin(neigh_list)],
    X='x',
    Y='y',
    exp='unique_region',
    hue=neighborhood_name,
    palette=palette,  # Apply the custom color palette
    invert_y=False,
    size=1,
    figsize=10
)


Plot "High" Cells

In [ ]:
#Subset of clusters to investigate/plot
neigh_list = [8,3,5,1,6]

#Same code as above, but just with plotting a subset of the clusters
s=sns.clustermap(fc.iloc[neigh_list,:], vmin =-3,vmax = 3,cmap = 'bwr',figsize=(10,4))

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(df.loc[df[neighborhood_name].isin(neigh_list)], X = 'x', Y='y', exp = 'unique_region',
               hue = neighborhood_name, invert_y=False, size = 2, figsize=8)

Plot the "Low" Clusters

In [ ]:
#Subset of clusters to investigate/plot
neigh_list = [0,4,7]

#Same code as above, but just with plotting a subset of the clusters
s=sns.clustermap(fc.iloc[neigh_list,:], vmin =-3,vmax = 3,cmap = 'bwr',figsize=(10,4))

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(df.loc[df[neighborhood_name].isin(neigh_list)], X = 'x', Y='y', exp = 'unique_region',
               hue = neighborhood_name, invert_y=False, size = 2, figsize=8)

Plot the "Medium" Clusters

In [ ]:
#Subset of clusters to investigate/plot
neigh_list = [9,2]

#Same code as above, but just with plotting a subset of the clusters
s=sns.clustermap(fc.iloc[neigh_list,:], vmin =-3,vmax = 3,cmap = 'bwr',figsize=(10,4))

#modify figure size aesthetics for each neighborhood
plt.rcParams["legend.markerscale"] = 15
figs = catplot2(df.loc[df[neighborhood_name].isin(neigh_list)], X = 'x', Y='y', exp = 'unique_region',
               hue = neighborhood_name, invert_y=False, size = 2, figsize=8)

In [92]:
n_conversion_10 = {
    0: '0',
    1: '1',
    2: '2',
    3: '3',
    4: '4',
    5: '5',
    6: '6',
    7: '7',
    8: '8',
    9: '9',

}
df['Probability_Bin_Cluster']=df[neighborhood_name].map(n_conversion_10)


In [ ]:
df

In [94]:
df.to_csv(r'C:\Users\k1van\Box\Home Folder kv120\Private\Research\PanOrgan\SO_Project2\intestine_all_probabilitybincluster.csv')